# Surrogates model for Dynamics and the MPC

In [ ]:
import torch as th, matplotlib.pyplot as plt, numpy as np
π = float(th.pi)
from torch.nn import Module, Parameter
import torch.nn.functional as F
from tqdm import tqdm
from matplotlib.ticker import MultipleLocator, FuncFormatter


# set dafault matplolib params
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.2
plt.style.use('dark_background')

In [ ]:
# define model dynamics, single pendulum, input = torque
def pendulum_dynamics(x, u):
    # x = [theta, theta_dot]
    # u = [tau] (torque)
    g = 9.81
    l = 1.0
    m = 1.0
    b = 0.1 # damping
    x_shape = x.shape # save original shape
    x = x.view(-1, 2)  # ensure x is of shape (n_par, 2)
    u = u.view(-1, 1)  # ensure u is of shape (n_par, 1)
    dtheta = x[:, 1]
    ddtheta = (u[:, 0] - m * g * l * th.sin(x[:, 0] + π) - b * x[:, 1]) / (m * l**2)
    return th.stack([dtheta, ddtheta], dim=1).view(x_shape)

def rk4_step(func, x, u, dt):
    k1 = func(x, u)
    k2 = func(x + 0.5 * dt * k1, u)
    k3 = func(x + 0.5 * dt * k2, u)
    k4 = func(x + dt * k3, u)
    return x + (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)

def euler_step(func, x, u, dt):
    k1 = func(x, u)
    return x + dt * k1

def sim(x0, us, n, dt):
    xs = th.zeros((n, *x0.shape))
    assert us.shape[0] == n-1, f'us should have shape ({n-1}, u_dim), but got {us.shape}'
    xs[0] = x0
    for i in range(1, n):
        xs[i] = rk4_step(pendulum_dynamics, xs[i-1], us[i-1], dt)
    return xs

def anim(xs):
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML, display
    fig, ax = plt.subplots()
    line, = ax.plot([], [], 'o-')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')

    def update(frame):
        x = xs[frame]
        line.set_data([0, th.cos(x[0]+π/2)], [0, th.sin(x[0]+π/2)])
        return line,

    ani = FuncAnimation(fig, update, frames=len(xs), blit=True, interval=3*100/6)
    # ani = FuncAnimation(fig, update, frames=len(xs), blit=True, interval=100)
    plt.close(fig)
    return display(HTML(ani.to_jshtml()))

def pi_formatter(x, pos):
    frac = x / np.pi
    if np.isclose(frac, 0):
        return "0"
    elif np.isclose(frac, 0.5):
        return r"$\frac{\pi}{2}$"
    elif np.isclose(frac, 1):
        return r"$\pi$"
    elif np.isclose(frac, -0.5):
        return r"$-\frac{\pi}{2}$"
    elif np.isclose(frac, -1):
        return r"$-\pi$"
    else:
        return r"${:.2f}\pi$".format(frac)
    
def my_plot(xs, dec=10):
    # plot trajectory in phase space
    plt.figure(figsize=(8, 4))
    plt.plot(xs[:, 0].numpy(), xs[:, 1].numpy())

    plt.gca().xaxis.set_major_locator(MultipleLocator(π/8))
    plt.gca().xaxis.set_major_formatter(FuncFormatter(pi_formatter))
    plt.xlabel('Theta (rad)')
    plt.ylabel('Omega (rad/s)')
    plt.title('Phase Space Trajectory')

    plt.tight_layout()
    plt.show()

    anim(xs[::dec])


## Let's start from the dynamics

In [ ]:
TH = 0.1 # [s] time horizon 
DT = 0.01 # time step
N = int(TH / DT) # prediction horizon
MAX_TORQUE = 2.0 # max torque
MAX_ANGLE = π # max angle
MAX_OMEGA = 8.0 # max angular velocity 

print(f"Prediction horizon: N={N}, Time step: DT={DT}s, Time horizon: TH={TH}s")

In [ ]:
# dataset
class DS(th.utils.data.Dataset):
    def __init__(self, n_samples):
        self.n_samples = n_samples
        θs = th.rand(n_samples) * 2 * MAX_ANGLE - MAX_ANGLE
        ωs = th.rand(n_samples) * 2 * MAX_OMEGA - MAX_OMEGA
        self.x0s = th.stack([θs, ωs], dim=1)
        self.us = th.rand(n_samples, N-1) * 2 * MAX_TORQUE - MAX_TORQUE
        # self.us = th.zeros(n_samples, N-1)
        self.x1s = th.zeros_like(self.x0s) # final states
        for i in tqdm(range(n_samples), desc="Generating dataset", leave=False):
            self.x1s[i] = sim(self.x0s[i], self.us[i], N, DT)[-1]
    def __len__(self):
        return self.n_samples
    def __getitem__(self, idx):
        return self.x0s[idx], self.us[idx], self.x1s[idx]

In [ ]:
# define the architecture
from torch.nn import Module, Parameter, Linear, Sequential
class Swish(th.nn.Module):
    def __init__(self):
        super(Swish, self).__init__()
        self.β = Parameter(th.tensor(1.0))
    def forward(self, x):
        return x * th.sigmoid(self.β * x)

class DynSurr(Module):
    def __init__(self, x_dim=2, u_dim=N-1, x_emb=4, u_emb=3): # action=input=u
        super(DynSurr, self).__init__()
        self.enc_x = Sequential(
            Linear(x_dim, 16), Swish(),
            Linear(16, x_emb), Swish(),
        )
        self.enc_u = Sequential(
            Linear(u_dim, 16), Swish(),
            Linear(16, u_emb), Swish(),
        )
        self.core = Sequential(
            Linear(x_emb + u_emb, 32), Swish(),
            Linear(32, 32), Swish(),
            Linear(32, x_dim), Swish(),
        )
    def forward(self, x, u):
        x = self.enc_x(x)
        u = self.enc_u(u)
        xu = th.cat([x, u], dim=-1)
        return self.core(xu) # new state


In [ ]:
# create dataset
ds = DS(10000)
dl = th.utils.data.DataLoader(ds, batch_size=32, shuffle=True)

In [ ]:
# train
ep = 40
model = DynSurr()
opt = th.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = th.nn.MSELoss()
for e in range(ep):
    total_loss = 0.0
    for x0s, us, x1s in dl:
        # print(f'x0s: {x0s.shape}, us: {us.shape}, x1s: {x1s.shape}')
        pred_x1s = model(x0s, us)
        loss = loss_fn(pred_x1s, x1s)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * x0s.shape[0]
    avg_loss = total_loss / len(ds)
    print(f"Epoch {e+1}/{ep}, Loss: {avg_loss:.6f}")

In [ ]:
# test the dynamics
def surr_step(x, u, model=model):
    with th.no_grad():
        return model(x.unsqueeze(0), u.unsqueeze(0)).squeeze(0)

x0 = th.rand(2) * 2 * th.tensor([MAX_ANGLE, MAX_OMEGA]) - th.tensor([MAX_ANGLE, MAX_OMEGA])
u = th.ones(N-1) * 5
x1t = sim(x0, th.zeros(N-1), N, DT)[-1] # true dynamics
x1p = surr_step(x0, th.zeros(N-1), model) # predicted dynamics

print(f"Initial state: {x0}")
print(f"True next state:      {x1t}")
print(f"Predicted next state: {x1p}")

In [ ]:
# simulate using surrogate model
n_sim = 300
us = th.zeros(N-1) # no torque
# us = th.ones(N-1) * .6
xsp = th.zeros((n_sim, 2))
xst = th.zeros((n_sim, 2))
xsp[0] = x0
xst[0] = x0
for i in range(1, n_sim):
    xsp[i] = surr_step(xsp[i-1], us, model)
    xst[i] = sim(xst[i-1], us, N, DT)[-1]

plt.figure(figsize=(12, 6))
plt.plot(xst[:, 0].numpy(), xst[:, 1].numpy(), label='True dynamics', alpha=0.5)
plt.plot(xsp[:, 0].numpy(), xsp[:, 1].numpy(), label='Surrogate model', alpha=0.5)
plt.show()

my_plot(xsp, dec=1)
my_plot(xst, dec=1)